# AI Governance Regulatory and Control Crosswalk — Colab Demo

This notebook drives the same `crosswalk/` package used by `cli.py`, just interactively.

**Run the cells in order.** Step 1 gets the code into this Colab session — use **either** Option A (clone from GitHub, once you've pushed the repo) **or** Option B (upload the zip directly). Don't run both.

## Step 1A — Clone from GitHub (recommended)

In [3]:
# Replace with your actual GitHub repo URL once it's pushed

!git clone https://github.com/MbowMiamiDade/ai-governance-crosswalk.git

%cd ai-governance-crosswalk

Cloning into 'ai-governance-crosswalk'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 26 (delta 2), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 22.08 KiB | 370.00 KiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/ai-governance-crosswalk/ai-governance-crosswalk


## Step 1B — Or upload the zip directly (alternative, if you don't have git set up locally)

In [4]:
# from google.colab import files

# uploaded = files.upload()  # select ai-governance-crosswalk.zip when prompted

In [5]:
# import zipfile



# zip_name = list(uploaded.keys())[0]

# with zipfile.ZipFile(zip_name, "r") as z:

#    z.extractall(".")



# %cd ai-governance-crosswalk

## Step 2 — Install dependencies

In [6]:
!pip install -q -r requirements.txt

## Step 3 — Load the data

In [7]:
from crosswalk.loader import load_all

from crosswalk.search import by_concept, by_free_text, concept_matrix_markdown



data = load_all()

print(f"Loaded {len(data.frameworks)} frameworks and {len(data.obligations)} obligations.")

Loaded 4 frameworks and 50 obligations.


## Step 4 — List the governance concepts

In [8]:
import pandas as pd



concept_df = pd.DataFrame(

    [(c.id, c.name, c.description.strip()) for c in data.concepts.values()],

    columns=["id", "name", "description"],

).sort_values("name")

concept_df

,id,name,description
11,bias_fairness,Bias & Fairness,"Requirements to test for, mitigate, and monito..."
8,change_management,Change Management,Controls over how models and AI systems are mo...
2,data_quality_governance,Data Quality & Data Governance,"Requirements around the quality, provenance, r..."
10,documentation_recordkeeping,Documentation & Recordkeeping,Requirements to produce and retain technical d...
0,governance_accountability,Governance & Accountability,"Clear ownership, roles, responsibilities, and ..."
3,human_oversight,Human Oversight,Requirements that a human can meaningfully und...
9,incident_response,Incident Response & Post-Market Monitoring,"Processes for detecting, reporting, and remedi..."
5,model_validation_testing,Model Validation & Testing,"Independent review, conceptual soundness check..."
6,monitoring_drift,Ongoing Monitoring & Drift,Post-deployment monitoring of a model's perfor...
1,risk_assessment,Risk Assessment & Classification,"Identifying, tiering, and classifying AI syste..."


## Step 5 — Look up a concept across all frameworks

This is the core "crosswalk" behavior: pick a governance concern, see every framework's obligation for it.

In [9]:
concept_to_check = "monitoring_drift"  # try: vendor_third_party_risk, bias_fairness, transparency_explainability, etc.



results = by_concept(data, concept_to_check)

for o in results:

    print(f"[{o.framework_name}] {o.citation} — {o.title}")

    print(f"  {o.summary.strip()}\n")

[CRI Financial Services AI Risk Management Framework (FS AI RMF)] FS AI RMF MS-2.4.1 — Monitoring and anomaly detection
  The organization monitors deployed AI systems in production using manual and automated tools, including statistical quality control methods, to detect and alert on performance deviations or anomalies.

[EU Artificial Intelligence Act (Regulation (EU) 2024/1689)] Article 15 — Accuracy, robustness, and cybersecurity
  High-risk systems must achieve appropriate levels of accuracy and be resilient to errors, faults, and attempts at manipulation.

[EU Artificial Intelligence Act (Regulation (EU) 2024/1689)] Article 26 — Obligations of deployers of high-risk AI systems
  Deployers must use systems per instructions, monitor operation, and keep logs, extending obligations beyond the original provider.

[EU Artificial Intelligence Act (Regulation (EU) 2024/1689)] Article 72 — Post-market monitoring
  Providers must implement a system to actively collect and review performanc

## Step 6 — Free-text search

TF-IDF cosine similarity over title + summary + concept names. No internet or model download required — runs fully offline, which matters since this needs to work the same in Colab as on GitHub Actions CI.

In [10]:
query = "monitoring drift"



scored_results = by_free_text(data, query, top_n=6)

for scored in scored_results:

    o = scored.obligation

    print(f"[{o.framework_name}] {o.citation} — {o.title}  (match: {scored.score:.2f})")

    print(f"  {o.summary.strip()}\n")

[NIST AI Risk Management Framework (AI RMF 1.0)] MANAGE 4.1 — Post-deployment monitoring feeds back into risk management  (match: 0.38)
  Lessons learned from monitoring and incidents are captured and used to improve the AI RMF process itself over time.

[2026 Interagency Model Risk Management Guidance (SR 26-2 / OCC Bulletin 2026-13 / FDIC FIL-15-2026)] SR 26-2, Section V — Ongoing model monitoring  (match: 0.33)
  Models are monitored for continued performance as products, exposures, or market conditions change, with a monitoring plan and defined procedures for responding to detected deterioration.

[EU Artificial Intelligence Act (Regulation (EU) 2024/1689)] Article 72 — Post-market monitoring  (match: 0.30)
  Providers must implement a system to actively collect and review performance data from a deployed high-risk system, with a written plan describing data sources and how findings feed back into risk management.

[NIST AI Risk Management Framework (AI RMF 1.0)] GOVERN 1.5 — Ongoi

## Step 7 — Full concept x framework matrix

In [11]:
from IPython.display import Markdown, display



display(Markdown(concept_matrix_markdown(data)))

| Concept | CRI Financial Services AI Risk Management Framework (FS AI RMF) | EU Artificial Intelligence Act (Regulation (EU) 2024/1689) | NIST AI Risk Management Framework (AI RMF 1.0) | 2026 Interagency Model Risk Management Guidance (SR 26-2 / OCC Bulletin 2026-13 / FDIC FIL-15-2026) |
|---|---|---|---|---|
| Bias & Fairness | FS AI RMF MS-2.11.1 | Article 10 | MAP 5.1<br>MEASURE 2.11 | — |
| Change Management | FS AI RMF MG-2.4.3 | — | MANAGE 2.2 | SR 26-2, Section IV<br>SR 26-2, Section V |
| Data Quality & Data Governance | FS AI RMF GV-1.1.6 | Article 10 | — | SR 26-2, Section IV |
| Documentation & Recordkeeping | FS AI RMF GV-1.1.6<br>FS AI RMF GV-1.6.1 | Article 9<br>Articles 11-12<br>Article 13<br>Article 26 | GOVERN 1.1<br>MAP 1.1<br>MANAGE 4.1 | SR 26-2, Section IV<br>SR 26-2, Section VI |
| Governance & Accountability | FS AI RMF GV-2.3.3<br>FS AI RMF GV-1.6.1 | Article 6 / Annex III<br>Article 9 | GOVERN 1.1<br>GOVERN 1.5<br>GOVERN 6.1 | SR 26-2, Section III<br>SR 26-2, Section VI |
| Human Oversight | FS AI RMF MP-3.5.1<br>FS AI RMF MG-2.4.3 | Article 14 | MEASURE 2.13<br>MANAGE 2.4 | — |
| Incident Response & Post-Market Monitoring | FS AI RMF GV-4.3.2 | Article 72<br>Article 73 | MANAGE 2.4<br>MANAGE 4.1 | — |
| Model Validation & Testing | FS AI RMF MS-2.9.1<br>FS AI RMF MS-2.5.1 | Article 15 | MAP 2.3<br>MEASURE 2.3<br>MEASURE 2.9<br>MEASURE 2.11<br>MEASURE 2.13 | SR 26-2, Section III<br>SR 26-2, Section IV<br>SR 26-2, Section V<br>SR 26-2, Section VII |
| Ongoing Monitoring & Drift | FS AI RMF MS-2.4.1 | Article 15<br>Article 26<br>Article 72 | GOVERN 1.5<br>MEASURE 2.3<br>MANAGE 2.2<br>MANAGE 4.1 | SR 26-2, Section V |
| Risk Assessment & Classification | FS AI RMF GV-1.3.1 | Article 6 / Annex III<br>Article 9 | MAP 1.1<br>MAP 2.3<br>MAP 5.1 | SR 26-2, Section III |
| Transparency & Explainability | FS AI RMF MS-2.9.1 | Article 13<br>Article 50 | MEASURE 2.8<br>MEASURE 2.9 | — |
| Vendor & Third-Party Risk | FS AI RMF GV-6.1.4 | Article 26 | GOVERN 6.1 | SR 26-2, Section VII |

## Step 8 — Run the data-integrity tests

These aren't testing AI behavior — they're testing that the YAML files are internally consistent (no obligation references a concept ID that doesn't exist, every obligation has required fields, etc.).

In [12]:
!pytest -q

..........                                                               [100%]
10 passed in 2.90s


---
**Scope and methodology:** crosswalk mappings identify conceptual alignment across selected governance requirements and controls — they do not imply legal or regulatory equivalence. Notably, the 2026 Interagency Model Risk Management Guidance (SR 26-2 / OCC Bulletin 2026-13 / FDIC FIL-15-2026) explicitly excludes generative AI and agentic AI models from its scope; its principles apply to traditional quantitative models and non-generative, non-agentic AI models. See the README for the full framework list and sourcing notes.